### RAG Pipelines - Data Ingestion to Vector Database

In [24]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path


In [25]:
### Read all the pdf inside the data/pdf folder and load them into a list of documents

def process_pdf_files(pdf_folder_path: str):
    """
    Process all PDF files in the specified folder and return a list of documents.

    Args:
        pdf_folder_path (str): The path to the folder containing PDF files.

    Returns:
        list: A list of Document objects.
    """
    # Create a list to store the loaded documents
    documents = []

    # Iterate over all files in the specified folder
    for filename in os.listdir(pdf_folder_path):
        print(f"Processing file: {filename}")  # Debugging statement
        if filename.endswith(".pdf"):
            # Create the full path to the PDF file
            file_path = os.path.join(pdf_folder_path, filename)

            # Load the PDF file
            loader = PyPDFLoader(file_path)
            pdf_documents = loader.load()

            # Add the loaded documents to the list
            documents.extend(pdf_documents)
            print(f"Loaded {len(pdf_documents)} documents from {filename}")  # Debugging statement


    return documents
    print(f"Total documents loaded: {len(documents)}")  # Debugging statement
    all_documents = process_pdf_files("../data")

In [26]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 2 PDF files to process

Processing: Course_Master_and_Serial_User_Manual (1).pdf
  ✓ Loaded 25 pages

Processing: Reference_Question_Paper_86.pdf
  ✓ Loaded 1 pages

Total documents loaded: 26


In [27]:
all_pdf_documents

[Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-09-27T19:49:35+00:00', 'author': 'Un-named', 'moddate': '2026-09-27T19:49:35+00:00', 'source': '..\\data\\pdf\\Course_Master_and_Serial_User_Manual (1).pdf', 'total_pages': 25, 'page': 0, 'page_label': '1', 'source_file': 'Course_Master_and_Serial_User_Manual (1).pdf', 'file_type': 'pdf'}, page_content='1 \n \n \n \n \n \n \n \n \n \n \n \n \n \n                        \n               LMS USER MANUAL \n  Learning Management System — Consolidated User Guide \n \nMilitary College of Material Management'),
 Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-09-27T19:49:35+00:00', 'author': 'Un-named', 'moddate': '2026-09-27T19:49:35+00:00', 'source': '..\\data\\pdf\\Course_Master_and_Serial_User_Manual (1).pdf', 'total_pages': 25, 'page': 1, 'page_label': '2', 'source_file': 'Course_Master_and_Serial_User_Manual (1).pdf', 'f

In [28]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [29]:
chunks=split_documents(all_pdf_documents, chunk_size=1000, chunk_overlap=200)
chunks

Split 26 documents into 31 chunks

Example chunk:
Content: 1 
 
 
 
 
 
 
 
 
 
 
 
 
 
                        
               LMS USER MANUAL 
  Learning Management System — Consolidated User Guide 
 
Military College of Material Management...
Metadata: {'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-09-27T19:49:35+00:00', 'author': 'Un-named', 'moddate': '2026-09-27T19:49:35+00:00', 'source': '..\\data\\pdf\\Course_Master_and_Serial_User_Manual (1).pdf', 'total_pages': 25, 'page': 0, 'page_label': '1', 'source_file': 'Course_Master_and_Serial_User_Manual (1).pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-09-27T19:49:35+00:00', 'author': 'Un-named', 'moddate': '2026-09-27T19:49:35+00:00', 'source': '..\\data\\pdf\\Course_Master_and_Serial_User_Manual (1).pdf', 'total_pages': 25, 'page': 0, 'page_label': '1', 'source_file': 'Course_Master_and_Serial_User_Manual (1).pdf', 'file_type': 'pdf'}, page_content='1 \n \n \n \n \n \n \n \n \n \n \n \n \n \n                        \n               LMS USER MANUAL \n  Learning Management System — Consolidated User Guide \n \nMilitary College of Material Management'),
 Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-09-27T19:49:35+00:00', 'author': 'Un-named', 'moddate': '2026-09-27T19:49:35+00:00', 'source': '..\\data\\pdf\\Course_Master_and_Serial_User_Manual (1).pdf', 'total_pages': 25, 'page': 1, 'page_label': '2', 'source_file': 'Course_Master_and_Serial_User_Manual (1).pdf', 'f

### EMbedding And VectorStoreDB

In [30]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

e:\YTRAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [31]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4959.97it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\vipin\AppData\Local\Temp\ipykernel_21028\2964522620.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


In [32]:
class VectorStore:
    """Handles document embeddings in a ChromaDB Vector Store"""

    def __init__(
        self,
        collection_name: str = "pdf_documents",
        persist_directory: str = "../data/vector_db"
    ):
        """
        Initialize the ChromaDB vector store.

        Args:
            collection_name: Name of the collection.
            persist_directory: Directory where ChromaDB stores data.
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None

        self._initialize_store()

    def _initialize_store(self):
        """Initialize the persistent ChromaDB client and collection."""

        try:
            # Create directory if it doesn't exist
            os.makedirs(self.persist_directory, exist_ok=True)

            # Current ChromaDB API
            self.client = chromadb.PersistentClient(
                path=self.persist_directory
            )

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={
                    "description": "Collection for PDF document embeddings"
                }
            )

            print(
                f"Vector store initialized with collection: "
                f"{self.collection_name}"
            )

            print(
                f"Existing Documents in collection: "
                f"{self.collection.count()}"
            )

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(
        self,
        documents: List[Any],
        embeddings: np.ndarray
    ):
        """
        Add documents and their embeddings to ChromaDB.

        Args:
            documents: List of LangChain Document objects.
            embeddings: Numpy array containing embeddings.
        """

        if len(documents) != len(embeddings):
            raise ValueError(
                "Number of documents and embeddings must match"
            )

        print(
            f"Adding {len(documents)} documents to the vector store..."
        )

        # Generate unique IDs
        ids = [
            str(uuid.uuid4())
            for _ in range(len(documents))
        ]

        # Extract metadata
        metadatas = [
            doc.metadata
            for doc in documents
        ]

        # Extract document text
        texts = [
            doc.page_content
            for doc in documents
        ]

        # Add to ChromaDB
        self.collection.add(
            ids=ids,
            metadatas=metadatas,
            embeddings=embeddings.tolist(),
            documents=texts
        )

        print(
            f"Added {len(documents)} documents to the vector store."
        )

        print(
            f"Total Documents in collection after addition: "
            f"{self.collection.count()}"
        )


vectorstore = VectorStore()
vectorstore

Vector store initialized with collection: pdf_documents
Existing Documents in collection: 0


In [35]:
### Convert the chunks into embeddings and add them to the vector store
texts=[doc.page_content for doc in chunks]

## Generate embedding
embeddings=embedding_manager.generate_embeddings(texts)

## Store int the vector store
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 31 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.36it/s]

Generated embeddings with shape: (31, 384)
Adding 31 documents to the vector store...
Added 31 documents to the vector store.
Total Documents in collection after addition: 31
